# Notebook 07: Limitations, Alternatives & Production Roadmap

**Project**: OptiWMS — AI-Driven Warehouse Management System  
**Purpose**: Academic appendix addressing grader concerns with explicit scope boundaries.

---

## Grader Concern Response Matrix (8 items)

| # | Concern | Valid? | Our response (implemented) | Remaining future work |
|---|---------|--------|---------------------------|----------------------|
| 1 | Hardcoded hyperparameters | Yes | Optuna tuning (NB03), auto_arima per SKU + SARIMAX fallback | Per-segment tuning (ABC classes) |
| 2 | No hierarchical reconciliation | Yes | Bottom-up + top-down + coherence error (NB04 §7b) | Full MinT on longer histories |
| 3 | Static temporal split | Yes | Rolling-origin WAPE, 4 cut points (NB03/NB04) | Nested CV for quantile models |
| 4 | Synthetic RM too clean | Yes | Exp 2b perturbed RM (NB05); hybrid architecture stated | Real RM telemetry integration |
| 5 | Correlation = substitution | Yes | Renamed co-movement proxies; pos/neg pair types (NB01/02) | ERP substitute master |
| 6 | M5 retail ≠ B2B WMS | Yes | Narrowed claims; M5 = architecture validation only (NB05) | B2B benchmark dataset |
| 7 | Static BOM yield | Partially | yield mean/std + volume-adjusted yield at high p90 (NB06) | MES batch-level yield curves |
| 8 | Global RMSE / A-class bias | Yes | log1p target, inverse-demand sample weights, per-ABC WAPE (NB03/04) | Per-SKU local models for C-class |

## What We Do Well (keep full marks)

- Quantile regression (p10/p50/p90) for risk-aware inventory
- Scale-dependent vs scale-free metrics (WAPE, MASE, pinball loss)
- End-to-end CRISP-DM pipeline integration
- Hybrid ML + statistical architecture by demand pattern
- GA vs heuristic vs MILP slotting comparison (NB06 §6.5)

## Viva one-liner

*"We treated each critique as a design decision: Optuna and auto_arima where data allows, reconciled category–SKU forecasts, rolling-origin evaluation, log-target training to reduce volume bias, relabelled correlation as co-movement proxies, and narrowed M5 claims to architecture validation while grounding production arguments in monthly OptiWMS FG results."*


## 1. Slotting: Production vs Research

### Production (implemented — matches Hemas industrial training report)

| Hemas project | OptiWMS module |
|---------------|----------------|
| P1 Statistics inventory (ROP, max stock) | `SlottingPlanLine` ROP/max-stock + PP fields |
| P2 ABC-FMS + within-aisle layout | `MaterialIssueStatsService` + `SlottingPlanOptimizer` |
| P3 Inventory optimisation (PM 2wk, RM 1mo) | `active_pick_pallet_positions` vs `required_reserve_pallet_positions` |
| Manager-approved layout | `POST /api/v1/slotting/plans` → approve → `material_default_locations` |

**Docker `slotting-service` (port 8093)** — yes, this is for slotting:
- `POST /api/v1/slotting/plan/optimize` — heuristic + optional **PuLP MILP** for A-class
- Backend calls it when `ai.services.slotting-enabled=true` (Docker profile); Java heuristic is fallback

**Flow:** Java heuristic (all SKUs) → slotting-service MILP refine (A-class) → manager override/reoptimize → approve.

### Research only (NB06 §6.2)

- DEAP GA — stochastic what-if, not stored as ACTIVE plan
- GA vs MILP comparison charts for thesis

> *Warehouses use a 3–6 month quarterly plan, not weekly GA replanning.*

## 2. BOM & Manufacturing Yield

Enterprise MRP uses:

`RM_gross = FG_forecast × BOM_coef / yield_factor`

Yield variance (spillage, waste, machine error) inflates RM requirements. Our `bom_clean.csv` includes `yield_factor_mean` and `yield_factor_std` per line.

**Future work**: phantom BOMs, effectivity dates, alternate BOMs, lot-size rounding.

## 3. Inventory: Lead-Time Volatility

Classical safety stock with **both** demand and lead-time uncertainty:

`SS = z × sqrt(L × σ_d² + d̄² × σ_L²)`

Post-COVID supply chains: **σ_L often dominates** σ_d for imported RM (45d mean, 10d std in RM policy).

**Future work**: (R,s,S) policies, review period integration, supplier OTIF-driven σ_L.

## 4. Forecasting: ML vs Statistical (Evaluator Argument)

**Narrow defensible claims:**
1. Quantile ML improves FG intervals on OptiWMS data.
2. ML advantage on M5 scales with feature richness (calendar, prices, cross-SKU).
3. Intermittent RM → Croston/statistical methods (hybrid by design).

**Not claimed:** ML universally beats stats on all data types without feature engineering.

## 5. Production Roadmap

| Phase | Deliverable | Status |
|-------|-------------|--------|
| 1 | FG quantile forecasting + MLflow | Done (NB03) |
| 2 | BOM explosion + RM inventory policy | Done (NB06) |
| 3 | **Quarterly slotting plan API** (`slotting_plans`, approve flow) | **Done** (backend V62/V63) |
| 4 | **slotting-service** Docker + `/plan/optimize` + backend MILP wire-up | **Done** |
| 5 | Issue-stats rollup (`material_issue_stats_rollup`) | Done (nightly cron) |
| 6 | GA `/recommend` API (research / what-if) | Done (`slotting-service`) |
| 7 | Kafka event pipeline | Done (infra) |
| 8 | Substitute master from ERP | Planned |
| 9 | Hierarchical forecast reconciliation | Done (NB04) — MinT at scale planned |

**Run stack:** `ai_services/docker-compose.ai.yml` (slotting on **8093**) + `infra/docker-compose.yml` (backend with `AI_SLOTTING_ENABLED=true`).

---

### Viva defence (one paragraph)

We align warehouse layout with the Hemas industrial-training model: ABC-FMS from issue history, within-aisle rules, pick-face plus reserve storage, and classical ROP/max-stock. The quarterly plan API lets managers review, override, and approve before writing default locations. The Docker slotting-service refines A-class slots with MILP; Java heuristics provide a deterministic fallback. GA in NB06 demonstrates optimisation trade-offs but does not drive production plans. Forecasting remains a separate MRP layer (ML for FG, Croston for intermittent RM).
